# Interoperabilidad con Qiskit y OpenQASM

Polypus no exige abandonar herramientas ya conocidas: acepta circuitos de Qiskit directamente, y puede exportar e importar OpenQASM 2.0, el formato de texto estándar para circuitos cuánticos.

## Ejecutar un circuito de Qiskit directamente

`run_quantum_circuit` acepta un `QuantumCircuit` de Qiskit tal cual, sin conversión previa:

In [1]:
import polypus
from qiskit import QuantumCircuit

qc_qiskit = QuantumCircuit(2)
qc_qiskit.h(0)
qc_qiskit.cx(0, 1)
qc_qiskit.measure_all()

result = polypus.run_quantum_circuit(qc_qiskit, shots=1000, infrastructure="local")
print(result.counts[0])

{'11': 515, '00': 485}


El resultado es el mismo estado de Bell del notebook anterior, `'00'` y `'11'` únicamente, construido esta vez con la API de Qiskit en lugar de con `polypus.Circuit`.

## Exportar a OpenQASM

Cualquier `Circuit` de Polypus se puede exportar a OpenQASM 2.0, el mismo formato de texto que usa Qiskit:

In [2]:
bell = polypus.Circuit(2).h(0).cx(0, 1).measure_all()
print(bell.to_qasm2())

OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0],q[1];
measure q -> c;



El resultado es texto plano, legible y portable a cualquier herramienta que entienda OpenQASM 2.0.

## Importar desde OpenQASM

`Circuit.from_qasm2` hace el camino inverso, y además garantiza una ida y vuelta exacta: exportar, importar y volver a exportar produce el mismo texto, byte a byte.

In [3]:
qasm_text = bell.to_qasm2()
bell_reimportado = polypus.Circuit.from_qasm2(qasm_text)

print(bell_reimportado.to_qasm2() == qasm_text)

True


Un circuito importado es un `Circuit` normal: se le pueden seguir encadenando puertas como a cualquier otro.

## Traer un circuito de Qiskit por QASM

`from_qasm2` también acepta el texto que produce Qiskit con `qasm2.dumps`, no solo el propio de Polypus. Es útil para convertir un circuito de Qiskit en un `Circuit` nativo, por ejemplo para seguir construyéndolo con el encadenado de métodos visto en `00a`:

In [4]:
from qiskit import qasm2

qasm_de_qiskit = qasm2.dumps(qc_qiskit)
circuito_nativo = polypus.Circuit.from_qasm2(qasm_de_qiskit)

print(circuito_nativo.to_qasm2())

OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0],q[1];
barrier q;
measure q -> c;



Al importar, Polypus normaliza algunas diferencias de notación entre herramientas: por ejemplo, varias formas equivalentes de expresar una puerta de un qubit se convierten todas a `u3`. El resultado sigue siendo el mismo circuito.

Además de OpenQASM, Polypus puede exportar a QIR, un formato de más bajo nivel pensado para compiladores. Queda fuera del alcance de esta serie, y se explica en el README principal.

## Visualizar un circuito

Polypus no tiene dibujado de circuitos propio, pero puede aprovecharse el de Qiskit: exportar a QASM e importar de vuelta con `qasm2.loads`, la función inversa de `qasm2.dumps` ya usada arriba.

In [ ]:
qc_para_dibujar = qasm2.loads(bell.to_qasm2())
qc_para_dibujar.draw("mpl")

El resultado es un diagrama del circuito: H sobre el qubit 0, CX entre los dos qubits, y la medida final de ambos. Requiere tener instalado `pylatexenc`; sin él, `draw()` sigue funcionando, pero como texto en lugar de imagen.

## Resumen

Este notebook ha mostrado varios caminos de interoperabilidad: ejecutar un circuito de Qiskit directamente, exportar un circuito de Polypus a OpenQASM, importar tanto el OpenQASM propio como el que genera Qiskit, y aprovechar el dibujado de Qiskit para visualizar un circuito de Polypus.

La siguiente sección trata sobre dónde se ejecutan estos circuitos: en local, o repartidos entre varios simuladores o QPUs.

## Siguiente paso

Continúa con: [`03a_ejecucion_local.ipynb`](03a_ejecucion_local.ipynb).